In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import math
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
# Load the uploaded file
df = pd.read_csv('新竹_2024.csv')

# Check unique values in '測項'
print("Unique Item Names:", df['測項'].unique())

# Melt the dataframe
df_melted = df.melt(id_vars=['測站', '日期', '測項'], var_name='Hour', value_name='Value')

# Drop '測站' as we only need the data (assuming it's all Hsinchu or we filter later)
# But let's check unique sites first
print("Unique Sites:", df['測站'].unique())

# Check the data type of 'Value' and see if there are non-numeric values
# We will convert to numeric later.

# Show head of melted data
print(df_melted.head())

Unique Item Names: ['AMB_TEMP' 'CH4' 'CO' 'NMHC' 'NO' 'NO2' 'NOx' 'O3' 'PM10' 'PM2.5'
 'RAINFALL' 'RH' 'SO2' 'THC' 'WD_HR' 'WIND_DIREC' 'WIND_SPEED' 'WS_HR']
Unique Sites: ['新竹']
   測站                   日期        測項 Hour Value
0  新竹  2024/01/01 00:00:00  AMB_TEMP   00  17.4
1  新竹  2024/01/01 00:00:00       CH4   00  2.06
2  新竹  2024/01/01 00:00:00        CO   00  0.38
3  新竹  2024/01/01 00:00:00      NMHC   00  0.07
4  新竹  2024/01/01 00:00:00        NO   00   0.3


In [8]:
# Define target features mapping
# Note: Renaming 'AMB_TEMP' to 'Temperature', 'WIND_SPEED' to 'WindSpeed' for clarity as per prompt
feature_map = {
    'PM2.5': 'PM2.5',
    'PM10': 'PM10',
    'O3': 'O3',
    'NO2': 'NO2',
    'AMB_TEMP': 'Temperature',
    'RH': 'Relative Humidity',
    'WIND_SPEED': 'Wind Speed'
}

target_items = list(feature_map.keys())
hour_cols = [f'{i:02d}' for i in range(24)]

# Melt
df_melted = df.melt(id_vars=['日期', '測項'], value_vars=hour_cols, var_name='Hour', value_name='Value')

# Convert Value to numeric, coercing errors
df_melted['Value'] = pd.to_numeric(df_melted['Value'], errors='coerce')

# Process Date
# The '日期' column format is 'YYYY/MM/DD HH:MM:SS'. We just need the YYYY/MM/DD part.
df_melted['Date'] = pd.to_datetime(df_melted['日期']).dt.date

# Create Timestamp
# Combine Date and Hour
# Hour is currently string '00', '01'...
df_melted['Timestamp'] = pd.to_datetime(df_melted['Date'].astype(str) + ' ' + df_melted['Hour'] + ':00:00')

# Pivot to wide format: Index=Timestamp, Columns=測項
df_pivot = df_melted.pivot(index='Timestamp', columns='測項', values='Value')

# Filter for Summer 2024 (June, July, August)
start_date = '2024-06-01'
end_date = '2024-08-31'
df_summer = df_pivot.loc[start_date:end_date].copy()

# Select only required columns and rename them
df_summer = df_summer[target_items]
df_summer.rename(columns=feature_map, inplace=True)

# Check for missing values
print("Missing values before cleaning:")
print(df_summer.isnull().sum())

# Fill missing values using linear interpolation (common for time series)
df_cleaned = df_summer.interpolate(method='linear', limit_direction='both')

# Check again
print("Missing values after cleaning:")
print(df_cleaned.isnull().sum())

# 1. Correlation Matrix
corr_matrix = df_cleaned.corr()

# Plot Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='viridis', fmt=".2f")
plt.title('Air Quality Feature Correlation Heatmap (Hsinchu, Summer 2024)')
plt.savefig('correlation_heatmap.png')
plt.close()

# 2. Scatter Plot with Marginal Histograms
# Prompt asks for one feature pair, e.g., PM2.5 vs Temperature
# Let's verify valid column names
print("Columns:", df_cleaned.columns)

# Using seaborn jointplot
g = sns.jointplot(data=df_cleaned, x='Temperature', y='PM2.5', kind='scatter', alpha=0.5, height=8)
g.fig.suptitle('PM2.5 vs Temperature with Marginal Histograms', y=1.02)
plt.savefig('scatter_plot.png')
plt.close()

# Save the cleaned data for future steps (Task 2 & 3)
df_cleaned.to_csv('Hsinchu_Summer_2024_Cleaned.csv')

print("Task 1 Completed. Files saved: correlation_heatmap.png, scatter_plot.png, Hsinchu_Summer_2024_Cleaned.csv")

Missing values before cleaning:
測項
PM2.5                18
PM10                 19
O3                   27
NO2                  41
Temperature          12
Relative Humidity     7
Wind Speed            2
dtype: int64
Missing values after cleaning:
測項
PM2.5                0
PM10                 0
O3                   0
NO2                  0
Temperature          0
Relative Humidity    0
Wind Speed           0
dtype: int64


C:\Users\yahoo\AppData\Local\Temp\ipykernel_26064\2497459644.py:61: UserWarning: Glyph 28204 (\N{CJK UNIFIED IDEOGRAPH-6E2C}) missing from font(s) DejaVu Sans.
  plt.savefig('correlation_heatmap.png')
C:\Users\yahoo\AppData\Local\Temp\ipykernel_26064\2497459644.py:61: UserWarning: Glyph 38917 (\N{CJK UNIFIED IDEOGRAPH-9805}) missing from font(s) DejaVu Sans.
  plt.savefig('correlation_heatmap.png')


Columns: Index(['PM2.5', 'PM10', 'O3', 'NO2', 'Temperature', 'Relative Humidity',
       'Wind Speed'],
      dtype='object', name='測項')
Task 1 Completed. Files saved: correlation_heatmap.png, scatter_plot.png, Hsinchu_Summer_2024_Cleaned.csv
